In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp

In [121]:
INPUT_DIR   = "ref_images"   # folder containing your reference images
VIS_DIR     = "ref_vis"      # where to save annotated images
OUT_NPZ     = "ref_blendshapes.npz"  # vectors + labels
MODEL_PATH  = "face_landmarker.task"  # update path

In [122]:
os.makedirs(VIS_DIR, exist_ok=True)

In [123]:
BaseOptions = mp.tasks.BaseOptions
Vision = mp.tasks.vision
VisionRunningMode = Vision.RunningMode
FaceLandmarker = Vision.FaceLandmarker
FaceLandmarkerOptions = Vision.FaceLandmarkerOptions

In [124]:
def draw_landmarks(img_bgr, landmarks_norm):
    """Draw small circles for each landmark onto a copy of the image."""
    output_image = img_bgr.copy()
    h, w = output_image.shape[:2]
    for lm in landmarks_norm:
        x = int(lm.x * w)
        y = int(lm.y * h)
        # color the dots purple with a radius of 2 pixels
        cv2.circle(output_image, (x, y), 2, (201, 141, 216), -1)
    return output_image

In [125]:
options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE, # IMAGE mode for stills
    num_faces=1,
    output_face_blendshapes=True, # <-- needed to store blendshape vectors
    output_facial_transformation_matrixes=False
)

In [126]:
def resize_letterbox(img, H=256, W=256, color=(0,0,0)):
    h, w = img.shape[:2]
    scale = min(W / w, H / h)
    nw, nh = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.full((H, W, 3), color, dtype=np.uint8)
    y0 = (H - nh) // 2
    x0 = (W - nw) // 2
    canvas[y0:y0+nh, x0:x0+nw] = resized
    return canvas

In [127]:
# all_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.jpeg')]
H, W = 800, 800

ref_cache = {}
for fn in sorted(os.listdir(INPUT_DIR)):
    if not fn.lower().endswith((".jpg",".jpeg",".png",".bmp",".webp")): 
        continue
    img = cv2.imread(os.path.join(INPUT_DIR, fn))
    if img is None: 
        print("skip:", fn); continue
    img_std = resize_letterbox(img, H, W)   # or resize_exact / center_crop_square_then_resize
    cv2.imwrite(os.path.join(INPUT_DIR, fn), img_std)
    ref_cache[fn] = img_std

In [128]:
labels, vecs = [], []
names = None

with FaceLandmarker.create_from_options(options) as landmarker:
    for fn in ref_cache:
        fp = os.path.join(INPUT_DIR, fn)
        img_bgr = cv2.imread(fp, cv2.IMREAD_COLOR)
        if img_bgr is None:
            print(f"[skip] failed to read: {fn}")
            continue

        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        res = landmarker.detect(mp_image)

        if not res.face_landmarks:        # no face detected
            print(f"[skip] no face: {fn}")
            continue

        # since num_faces=1, index 0 if present
        lm0 = res.face_landmarks[0]

        if not res.face_blendshapes:
            print(f"[skip] no blendshapes: {fn}")
            continue

        print({fn})
        cats = res.face_blendshapes[0]
        b = np.array([c.score for c in cats], dtype=np.float32)

        if names is None:
            names = [c.category_name for c in cats]

        annotated = draw_landmarks(img_bgr, lm0)
        cv2.imwrite(os.path.join(VIS_DIR, fn), annotated)

        labels.append(fn)
        vecs.append(b)
        print(f"[ok] {fn} → {len(b)} blendshape dims")

I0000 00:00:1762298544.550988 23963940 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 83.1), renderer: Apple M1
W0000 00:00:1762298544.551228 23963940 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1762298544.554824 24174188 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1762298544.565667 24174188 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


{'image1.jpeg'}
[ok] image1.jpeg → 52 blendshape dims
[skip] no face: image10.jpeg
{'image11.jpeg'}
[ok] image11.jpeg → 52 blendshape dims
{'image12.jpeg'}
[ok] image12.jpeg → 52 blendshape dims
{'image13.jpeg'}
[ok] image13.jpeg → 52 blendshape dims
{'image14.jpeg'}
[ok] image14.jpeg → 52 blendshape dims
{'image15.jpeg'}
[ok] image15.jpeg → 52 blendshape dims
{'image16.jpeg'}
[ok] image16.jpeg → 52 blendshape dims
{'image17.jpeg'}
[ok] image17.jpeg → 52 blendshape dims
{'image18.jpeg'}
[ok] image18.jpeg → 52 blendshape dims
[skip] no face: image19.jpeg
{'image2.jpeg'}
[ok] image2.jpeg → 52 blendshape dims
[skip] no face: image20.jpeg
{'image21.jpeg'}
[ok] image21.jpeg → 52 blendshape dims
{'image22.jpeg'}
[ok] image22.jpeg → 52 blendshape dims
{'image23.jpeg'}
[ok] image23.jpeg → 52 blendshape dims
{'image24.jpeg'}
[ok] image24.jpeg → 52 blendshape dims
{'image25.jpeg'}
[ok] image25.jpeg → 52 blendshape dims
{'image26.jpeg'}
[ok] image26.jpeg → 52 blendshape dims
{'image27.jpeg'}
[ok]

In [129]:
if not vecs:
    print("No references collected.")

In [130]:
vecs = np.vstack(vecs)  # [N, D]
labels = np.array(labels)

# L2-normalize (often helps for cosine/Euclidean matching later)
norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-8
vecs_norm = vecs / norms

# Save everything
np.savez(OUT_NPZ,
        labels=labels,
        vectors=vecs_norm,
        raw_vectors=vecs,     # optional: unnormalized copy
        blendshape_names=np.array(names))

print(f"\nSaved {len(labels)} references to {OUT_NPZ}")
print(f"Annotated images → {VIS_DIR}")
print(f"Blendshape dims: {vecs.shape[1]}")


Saved 27 references to ref_blendshapes.npz
Annotated images → ref_vis
Blendshape dims: 52
